In [1]:
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
from qdots_qll.models.models_scratch_for_drafting import SingleQDot3Params
from qdots_qll.models.game import true_pars
import qutip as qt
from functools import partial

In [2]:
true_pars

In [3]:
from qdots_qll.utils.povms import  sigmas_povm

In [4]:
Gs = np.array(
    [qt.identity(2), qt.sigmax(), qt.sigmay(), qt.sigmaz()]
) / np.sqrt(2)

In [5]:
rho0 = qt.rand_dm_ginibre(2).full()

In [254]:
eta/2

In [253]:
(U@Hnazir@U.T).round(3)

In [10]:
qt.ket2dm(qt.basis(2, 0))

In [6]:
m = SingleQDot3Params(sigmas_povm)

gamma_minus = float(true_pars[0])
gamma_plus = float(true_pars[1])
S_minus = float(true_pars[2])
S_plus = float(true_pars[3])
bigOmega = m.Omega
delta = float(m.make_system_hamiltonian()[0, 0])

eta = np.sqrt(delta**2 + bigOmega**2)

In [7]:
def rho_to_bloch(rho):
    return np.einsum("ijk,kj-> i", Gs, rho).real


def bloch_to_rho(v):
    return np.einsum("ijk,i-> jk", Gs, v)

In [8]:
def make_U(delta, bigOmega, eta):

    u00 = -(bigOmega / np.sqrt(2)) / (
        np.sqrt(bigOmega**2 + delta * (delta + eta))
    )
    u01 = (delta + eta) / (
        bigOmega * (np.sqrt(1 + (delta + eta) ** 2 / (bigOmega**2)))
    )
    u10 = (np.sqrt(1 + (delta) / (eta))) / (np.sqrt(2))
    u11 = 1 / (np.sqrt(1 + (delta + eta) ** 2 / (bigOmega**2)))
    # u00 = (delta - eta)/(delta) *np.sqrt(delta/eta + 1)
    # u01 = (np.sqrt(1-delta/eta))*(eta/delta + delta/bigOmega)
    # u10 = (np.sqrt(delta/eta + 1))
    # u11 = (delta*bigOmega)/(np.sqrt(eta*bigOmega*(delta**2 + eta*bigOmega)))
    U = np.array([[u00, u01], [u10, u11]])
    return U

In [9]:
U = make_U(delta, bigOmega, eta)

In [10]:
Hnazir = delta / 2 * qt.sigmaz() + bigOmega / 2 * qt.sigmax()
Hnazir = Hnazir.full()
Hnazir

In [11]:
rho0tilde = qt.ket2dm(qt.basis(2, 0)).full()
rho0 = U.T @ rho0tilde @ U
v = rho_to_bloch(rho0)

In [12]:
gamma_plus

In [13]:
(U.T @ Hnazir @ U).round(4)

In [14]:
def evolve_v(
    v,
    t,
    gamma_minus,
    gamma_plus,
    S_minus,
    S_plus,
    bigOmega,
    eta,
    delta,
):
    # v0, v1, v2, v3 = v

    # v0t = v0
    # v1t = (
    #     v1
    #     * np.cos(
    #         eta * t - (bigOmega**2 * t * ((S_minus) - (S_plus))) / (4 * eta**2)
    #     )
    #     + v2
    #     * np.sin(
    #         eta * t - (bigOmega**2 * t * ((S_minus) - (S_plus))) / (4 * eta**2)
    #     )
    # ) * np.exp(
    #     -1
    #     * (t * (delta**2 * (gamma_minus) + bigOmega**2 * (gamma_plus)))
    #     / (2 * eta**2)
    # )
    # v2t = (
    #     v2
    #     * np.cos(
    #         eta * t - (bigOmega**2 * t * ((S_minus) - (S_plus))) / (4 * eta**2)
    #     )
    #     - v1
    #     * np.sin(
    #         eta * t - (bigOmega**2 * t * ((S_minus) - (S_plus))) / (4 * eta**2)
    #     )
    # ) * np.exp(
    #     -1
    #     * (t * (delta**2 * (gamma_minus) + bigOmega**2 * (gamma_plus)))
    #     / (2 * eta**2)
    # )

    # v3t = np.exp(
    #     -1
    #     * (t * (delta**2 * (gamma_minus) + bigOmega**2 * (gamma_plus)))
    #     / (eta**2)
    # ) * v3 + (
    #     1
    #     - np.exp(
    #         -1
    #         * (t * (delta**2 * (gamma_minus) + bigOmega**2 * (gamma_plus)))
    #         / (eta**2)
    #     )
       
    # ) * v0*(-(delta**2) * (gamma_minus) + bigOmega**2 * (gamma_plus)) / (
    #     delta**2 * (gamma_minus) + bigOmega**2 * (gamma_plus)
    # )
    # return np.array([v0t, v1t, v2t, v3t])


        # v0, v1, v2, v3 = v

        # v0t = v0
        # v1t = (
        #     v1
        #     * np.cos(
        #         eta * t - (bigOmega**2 * t * ((S_minus) - (S_plus))) / (4 * eta**2)
        #     )
        #     + v2
        #     * np.sin(
        #         eta * t - (bigOmega**2 * t * ((S_minus) - (S_plus))) / (4 * eta**2)
        #     )
        # ) * np.exp(
        #     -1
        #     * (t * (bigOmega**2 * (gamma_minus) + bigOmega**2 * (gamma_plus)))
        #     / (4 * eta**2)
        # )
        # v2t = (
        #     v2
        #     * np.cos(
        #         eta * t - (bigOmega**2 * t * ((S_minus) - (S_plus))) / (4 * eta**2)
        #     )
        #     - v1
        #     * np.sin(
        #         eta * t - (bigOmega**2 * t * ((S_minus) - (S_plus))) / (4 * eta**2)
        #     )
        # ) * np.exp(
        #     -1
        #     * (t * (bigOmega**2 * (gamma_minus) + bigOmega**2 * (gamma_plus)))
        #     / (4 * eta**2)
        # )

        # v3t = np.exp(
        #     -1
        #     * (t * (bigOmega**2 * (gamma_minus) + bigOmega**2 * (gamma_plus)))
        #     / (2*eta**2)
        # ) * v3 + (
        #     1
        #     - np.exp(
        #         -1
        #         * (t * (bigOmega**2 * (gamma_minus) + bigOmega**2 * (gamma_plus)))
        #         / (2*eta**2)
        #     )
        
        # ) * v0*(-(bigOmega**2) * (gamma_minus) + bigOmega**2 * (gamma_plus)) / (
        #     bigOmega**2 * (gamma_minus) + bigOmega**2 * (gamma_plus)
        # )
        # return np.array([v0t, v1t, v2t, v3t])

        v0, v1, v2, v3 = v

        v0t = v0
        v1t = (
            v1
            * np.cos(
                eta * t - (bigOmega**2 * t * ((S_minus) - (S_plus))) / (4 * eta**2)
            )
            + v2
            * np.sin(
                eta * t - (bigOmega**2 * t * ((S_minus) - (S_plus))) / (4 * eta**2)
            )
        ) * np.exp(
            -1
            * (t * (bigOmega**2 * (gamma_minus) + bigOmega**2 * (gamma_plus)))
            / (4 * eta**2)
        )
        v2t = (
            v2
            * np.cos(
                eta * t - (bigOmega**2 * t * ((S_minus) - (S_plus))) / (4 * eta**2)
            )
            - v1
            * np.sin(
                eta * t - (bigOmega**2 * t * ((S_minus) - (S_plus))) / (4 * eta**2)
            )
        ) * np.exp(
            -1
            * (t * (bigOmega**2 * (gamma_minus) + bigOmega**2 * (gamma_plus)))
            / (4 * eta**2)
        )

        v3t = np.exp(
            -1
            * (t * (bigOmega**2 * (gamma_minus) + bigOmega**2 * (gamma_plus)))
            / (2*eta**2)
        ) * v3 + (
            1
            - np.exp(
                -1
                * (t * (bigOmega**2 * (gamma_minus) + bigOmega**2 * (gamma_plus)))
                / (2*eta**2)
            )
        
        ) * v0*(-(bigOmega**2) * (gamma_minus) + bigOmega**2 * (gamma_plus)) / (
            bigOmega**2 * (gamma_minus) + bigOmega**2 * (gamma_plus)
        )
        return np.array([v0t, v1t, v2t, v3t])

In [20]:
lambda_v = partial(
    evolve_v,
    gamma_minus=1 * gamma_minus/4 ,
    gamma_plus=1 * gamma_plus /4,
    S_minus=1 * S_minus,
    S_plus=1 * S_plus,
    bigOmega=bigOmega,
    eta=eta,
    delta=delta,
)

times = np.linspace(0, 70, 200)
evolved_rhos = np.array(
    list(
        [
            (
                lambda t: (
                    U.T
                    @ bloch_to_rho(
                        lambda_v(rho_to_bloch(U @ rho0tilde @ U.T), t)
                    )
                    @ U
                )
            )(t)
            for t in times
        ]
    )
)
povms_evolved = jax.vmap(
    lambda t: m.likelihood_particle(true_pars[1:], t, m.vec(rho0tilde))
)(times)
plt.plot(times, evolved_rhos[:, 1, 1], label="map")
plt.plot(times, povms_evolved[:, 4] * 3, label="nazir paper")
plt.legend()
plt.show()

In [38]:
k = 1

povm_evols = jax.vmap(lambda rho, proj: jnp.trace(rho@proj), in_axes=(0, None))(evolved_rhos,sigmas_povm[k]*3,)


plt.plot(times, povm_evols, label="map")
plt.plot(times, povms_evolved[:, k] * 3, label="nazir paper")
plt.legend()
plt.show()

In [22]:
sigmas_povm

In [3]:
plt

In [91]:
m.likelihood_particle?

In [90]:
m.__dir__()